# AK(3) stable-proof lanes — one box per Colab runtime

Prove **AK(3) is stably AC-trivial** by finding any certified chain to the trivial presentation.
Open this notebook in up to **5 high-RAM CPU runtimes** and set `BOX` to a different value in each:

| BOX | what it runs | ~time |
|---|---|---|
| **D1** | Lane D (plateau elimination) at scale, **textbook** form: harvest 500k-node balls for 10 z=w stabilizations, Lemma-11-eliminate every visited state -> fresh 2-gen presentations ~stAC~ AK3, greedy-solve the 12,000 shortest @50k | ~2-3h |
| **D2** | same, **rep** form | ~2-3h |
| **D3** | Lane D with the FULL ~95-word bank, both forms (harvest 200k, solve @25k) | ~5-8h |
| **B**  | StableSolver (substitution+stabilize+eliminate in one search) @800k, g<=3/4 | done / ~3h |
| **C**  | dumb stabilization baselines n=3/4/5 @0.8-2M + MITM outward @2M with ball dumps | done / ~4h |

The harvest hot path is now **numba-compiled (~3.7x/combo, gold-verified identical output)** and
uses **all cores** (auto-sized to RAM), so a box that was crawling on 4 cores now saturates the
runtime. Times above assume the full box; a resumed box only does what's left.

**Any `*** SOLVED ***` / verified cert = AK(3) is stably AC-trivial.** Everything is resumable:
if the runtime dies, just Run All again — finished combos are skipped (a combo is done once its
`cands_<form>_<word>.jsonl` exists on Drive, with or without its `.done` marker). Run **s5 any time**
(even before s4) to see how many combos are already done and confirm resume picked them up.
When a box finishes (or in the morning), share the Drive folder `ak3_stable_proof/` back.


In [ ]:
# ==================== s1  CONFIG  (edit ONLY this cell) ====================
BOX          = "D1"        # <-- D1 | D2 | D3 | B | C  (one per runtime)
QUICK_TEST   = False       # True = ~2-min end-to-end base case FIRST; then set False and rerun s4
WORKERS      = None        # None = auto from CPU count

REPO_URL     = "https://github.com/Avi161/ACSolverX.git"
BRANCH       = "test/stable-ac-moves"
REPO_DIR     = "ACSolverX"
UPDATE_REPO  = True
MOUNT_DRIVE  = True
print(f"config: BOX={BOX} QUICK_TEST={QUICK_TEST}")

In [ ]:
# ==================== s2  SETUP  (clone / pull / install / mount) ====================
import os, subprocess, sys
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-1500:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numpy numba psutil")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = f"/content/drive/MyDrive/ak3_stable_proof/{BOX}"
    else:
        OUT_DIR = f"/content/ak3_out/{BOX}"
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(".")
    OUT_DIR = os.path.abspath(f"./.scratch/colab_{BOX}")
os.makedirs(OUT_DIR, exist_ok=True)
NCPU = os.cpu_count()
print(f"repo={REPO_ROOT}  out={OUT_DIR}  cores={NCPU}")

In [ ]:
# ==================== s3  QUICK BASE CASE  (run once; ~2 min; must end 'complete') ====
if QUICK_TEST:
    script = os.path.join(REPO_ROOT, "experiments", "stable_ac", "ak3_stable_proof", "run_lanes.py")
    cmd = [sys.executable, script, "--box", BOX, "--out_dir", OUT_DIR + "_quick", "--quick"]
    if WORKERS: cmd += ["--workers", str(WORKERS)]
    print(" ".join(cmd)); subprocess.run(cmd, check=True)
    print("\nBASE CASE OK -> set QUICK_TEST=False and run s4")
else:
    print("QUICK_TEST=False -> skip (s4 is the real run)")

In [ ]:
# ==================== s4  RUN  (streams live; resumable after any disconnect) ==========
if not QUICK_TEST:
    script = os.path.join(REPO_ROOT, "experiments", "stable_ac", "ak3_stable_proof", "run_lanes.py")
    cmd = [sys.executable, script, "--box", BOX, "--out_dir", OUT_DIR]
    if WORKERS: cmd += ["--workers", str(WORKERS)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("QUICK_TEST=True -> flip it to False first")

In [ ]:
# ==================== s5  PROGRESS / STATUS  (run ANYTIME — even during s4) ============
# Shows how far the box has gotten. Run this BEFORE s4 to confirm resume sees prior work
# (HARVEST done/total should be > 0 if you re-uploaded earlier results to this Drive folder).
import glob, json, sys
sys.path.insert(0, os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator"))

def _util():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return psutil.cpu_percent(interval=0.3), vm.used / 1e9, vm.total / 1e9
    except Exception:
        return float("nan"), float("nan"), float("nan")

def tally_jsonl(path, key="solved"):
    n = s = 0
    for line in open(path):
        try: r = json.loads(line)
        except Exception: continue
        n += 1; s += int(bool(r.get(key)))
    return n, s

cpu, ram, rt = _util()
print(f"=== BOX {BOX} ===   CPU {cpu:.0f}%   RAM {ram:.0f}/{rt:.0f} GB   (cores={os.cpu_count()})\n")

if BOX in ("D1", "D2", "D3"):
    import ak3_words as aw
    HERO = ["xyx", "yxy", "xxx", "yyyy", "Xyxy", "YXyxy", "x", "y"]
    if BOX == "D1":   forms, words = ["textbook", "p25"],   HERO + ["r1", "r2"]
    elif BOX == "D2": forms, words = ["rep", "floorF"],     HERO + ["r1", "r2"]
    else:             forms, words = ["textbook", "rep"], [e["name"] for e in aw.build_word_bank()]
    total = len(forms) * len(words)
    laneD = os.path.join(OUT_DIR, "laneD")
    present = set(os.listdir(laneD)) if os.path.isdir(laneD) else set()
    done = 0; walls = []
    for f in forms:
        for w in words:
            if f"cands_{f}_{w}.jsonl" in present or f"cands_{f}_{w}.done" in present:
                done += 1
                if f"cands_{f}_{w}.done" in present:
                    try: walls.append(json.loads(open(os.path.join(laneD, f"cands_{f}_{w}.done")).read()).get("wall_s", 0))
                    except Exception: pass
    pct = 100 * done / total if total else 0
    avg = sum(walls) / len(walls) if walls else 0
    print(f"HARVEST : {done}/{total} combos done ({pct:.0f}%)   {total - done} remaining"
          + (f"   ~{avg:.0f}s/combo measured" if walls else ""))
    mj = os.path.join(laneD, "merged.jsonl")
    if os.path.exists(mj):
        print(f"MERGE   : merged.jsonl present ({sum(1 for _ in open(mj))} unique candidates)")
    sj = os.path.join(laneD, "solve.jsonl")
    if os.path.exists(sj):
        n, s = tally_jsonl(sj)
        print(f"SOLVE   : {n} candidates attempted, {s} SOLVED")
    certs = glob.glob(os.path.join(OUT_DIR, "certs", "laneD_*.json"))
    if certs:
        print(f"CERTS   : {len(certs)} ->", *map(os.path.basename, certs))
    print("\n" + (">>> harvest COMPLETE (merge/solve next)" if done == total
                   else "... harvest in progress — re-run this cell to refresh"))
else:
    for p in sorted(glob.glob(os.path.join(OUT_DIR, "box_*.jsonl"))):
        n, s = tally_jsonl(p)
        print(f"{os.path.basename(p)}: {n} configs done, {s} solved/hit")
        for line in open(p):
            r = json.loads(line)
            print("  ", json.dumps({k: r.get(k) for k in
                  ("night_id", "solved", "hit", "nodes", "min_total_len", "wall_s", "peak_rss_mb", "error")}))
print("\nIf anything says SOLVED / hit: share the Drive folder — that is the theorem.")
